# World Cup Accuracy Improvement Experiments

This notebook tests accuracy-improvement ideas against the same frozen 2010, 2014, 2018, and 2022 FIFA World Cup audit used by `09_world_cup_2010_2022_backtest.ipynb`.

It keeps the work notebook-first and self-contained: feature engineering, model fitting, validation, and report generation live here. The local repository does not contain betting odds, squad, injury, player-market-value, or FIFA-ranking datasets, so those methods are recorded as unavailable rather than fabricated.


In [1]:
from pathlib import Path
from math import exp, factorial
import itertools
import warnings

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression, PoissonRegressor
from sklearn.metrics import log_loss
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier

warnings.filterwarnings('ignore', category=FutureWarning)

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA = ROOT / 'data' / 'processed' / 'matches_with_elo.csv'
REPORTS = ROOT / 'reports'
YEARS = [2010, 2014, 2018, 2022]
SEED = 20260707
ORDERED = np.array(['H', 'D', 'A'])
LABELS = {'H': 0, 'D': 1, 'A': 2}

matches = pd.read_csv(DATA, parse_dates=['date']).sort_values(['date', 'match_id']).reset_index(drop=True)
required = {'match_id', 'date', 'home_team', 'away_team', 'home_score', 'away_score', 'result', 'tournament', 'neutral', 'home_elo', 'away_elo', 'elo_difference', 'elo_home_probability', 'elo_draw_probability', 'elo_away_probability'}
assert required <= set(matches.columns), required - set(matches.columns)

for unavailable in ['odds', 'injury', 'squad', 'player', 'market_value', 'fifa_rank']:
    assert not any(unavailable in path.name.lower() for path in (ROOT / 'data').rglob('*')), f'unexpected {unavailable} data found; wire it into the notebook explicitly'

def classify_tournament(name):
    text = str(name).lower()
    if 'qualif' in text:
        return 'qualifier'
    if str(name) == 'FIFA World Cup':
        return 'world_cup'
    markers = ['copa america', 'copa américa', 'african cup of nations', 'afc asian cup', 'uefa euro', 'european championship', 'gold cup', 'oceania', 'nations league']
    if any(marker in text for marker in markers):
        return 'continental'
    if text == 'friendly':
        return 'friendly'
    return 'other'

def add_world_cup_stage(frame):
    frame = frame.copy()
    frame['world_cup_year'] = np.where(frame['tournament'].eq('FIFA World Cup'), frame['date'].dt.year, np.nan)
    frame['wc_match_number'] = -1
    mask = frame['tournament'].eq('FIFA World Cup')
    frame.loc[mask, 'wc_match_number'] = frame.loc[mask].groupby(frame.loc[mask, 'date'].dt.year).cumcount()
    frame['wc_group_stage'] = (frame['wc_match_number'].between(0, 47)).astype(float)
    frame['wc_knockout_stage'] = (frame['wc_match_number'] >= 48).astype(float)
    return frame

def build_team_panel(frame):
    points = {'H': 1.0, 'D': 0.5, 'A': 0.0}
    home = pd.DataFrame({
        'match_id': frame.match_id, 'date': frame.date, 'team': frame.home_team, 'is_home': True,
        'goals_for': frame.home_score.astype(float), 'goals_against': frame.away_score.astype(float),
        'team_elo': frame.home_elo.astype(float), 'opponent_elo': frame.away_elo.astype(float),
        'result_points': frame.result.map(points).astype(float),
        'expected_points': frame.elo_home_probability + 0.5 * frame.elo_draw_probability,
    })
    away = pd.DataFrame({
        'match_id': frame.match_id, 'date': frame.date, 'team': frame.away_team, 'is_home': False,
        'goals_for': frame.away_score.astype(float), 'goals_against': frame.home_score.astype(float),
        'team_elo': frame.away_elo.astype(float), 'opponent_elo': frame.home_elo.astype(float),
        'result_points': 1.0 - frame.result.map(points).astype(float),
        'expected_points': frame.elo_away_probability + 0.5 * frame.elo_draw_probability,
    })
    panel = pd.concat([home, away], ignore_index=True).sort_values(['team', 'date', 'match_id'], kind='mergesort').reset_index(drop=True)
    grouped = panel.groupby('team', sort=False)
    panel['days_since_match'] = grouped.date.diff().dt.days.fillna(365).clip(0, 365)
    recent_counts = pd.Series(0.0, index=panel.index)
    for _, idx in panel.groupby('team', sort=False).groups.items():
        dates = panel.loc[idx, 'date'].to_numpy()
        counts = []
        for pos, current in enumerate(dates):
            prior = dates[:pos]
            counts.append(float(np.sum((current - prior) <= np.timedelta64(365, 'D'))))
        recent_counts.loc[idx] = counts
    panel['recent_matches_365'] = recent_counts
    for window in [5, 10]:
        for col in ['goals_for', 'goals_against', 'result_points']:
            panel[f'rolling_{window}_{col}'] = grouped[col].transform(lambda s, w=window: s.shift(1).rolling(w, min_periods=1).mean()).fillna({'goals_for': 1.25, 'goals_against': 1.25, 'result_points': 0.5}[col])
        actual = grouped.result_points.transform(lambda s, w=window: s.shift(1).rolling(w, min_periods=1).mean())
        expected = grouped.expected_points.transform(lambda s, w=window: s.shift(1).rolling(w, min_periods=1).mean())
        panel[f'rolling_{window}_form_vs_expected'] = (actual - expected).fillna(0.0)
        panel[f'rolling_{window}_elo_trend'] = (grouped.team_elo.shift(1) - grouped.team_elo.shift(1 + window)).fillna(0.0)
    return panel

def build_features(frame):
    frame = add_world_cup_stage(frame)
    frame['tournament_type'] = frame.tournament.map(classify_tournament)
    panel = build_team_panel(frame)
    value_cols = [c for c in panel.columns if c.startswith('rolling_')] + ['days_since_match', 'recent_matches_365']
    home = panel.loc[panel.is_home, ['match_id', *value_cols]].rename(columns={c: f'home_{c}' for c in value_cols})
    away = panel.loc[~panel.is_home, ['match_id', *value_cols]].rename(columns={c: f'away_{c}' for c in value_cols})
    out = frame.merge(home, on='match_id', how='left').merge(away, on='match_id', how='left')
    out['abs_elo_difference'] = out.elo_difference.abs()
    out['elo_similarity'] = np.exp(-out.abs_elo_difference / 200.0)
    out['elo_difference_sq'] = np.sign(out.elo_difference) * (out.elo_difference / 400.0) ** 2
    out['recent_goal_total_5'] = out.home_rolling_5_goals_for + out.away_rolling_5_goals_for
    out['recent_defence_total_5'] = out.home_rolling_5_goals_against + out.away_rolling_5_goals_against
    out['recent_form_gap_5'] = out.home_rolling_5_result_points - out.away_rolling_5_result_points
    out['recent_form_gap_10'] = out.home_rolling_10_result_points - out.away_rolling_10_result_points
    out['recent_attack_gap_5'] = out.home_rolling_5_goals_for - out.away_rolling_5_goals_for
    out['recent_defence_gap_5'] = out.home_rolling_5_goals_against - out.away_rolling_5_goals_against
    out['rest_gap'] = out.home_days_since_match - out.away_days_since_match
    out['activity_gap_365'] = out.home_recent_matches_365 - out.away_recent_matches_365
    return out.sort_values(['date', 'match_id']).reset_index(drop=True)

featured = build_features(matches)
assert featured.filter(regex='rolling_|days_since|recent_|elo_similarity|stage|gap').isna().sum().sum() == 0
featured.shape


(49493, 62)

In [2]:
BASE_FEATURES = ['elo_difference', 'neutral']
RICH_FEATURES = [
    'elo_difference', 'abs_elo_difference', 'elo_difference_sq', 'elo_similarity', 'neutral',
    'home_elo', 'away_elo', 'wc_group_stage', 'wc_knockout_stage',
    'home_rolling_5_goals_for', 'home_rolling_5_goals_against', 'home_rolling_5_result_points', 'home_rolling_5_form_vs_expected', 'home_rolling_5_elo_trend',
    'away_rolling_5_goals_for', 'away_rolling_5_goals_against', 'away_rolling_5_result_points', 'away_rolling_5_form_vs_expected', 'away_rolling_5_elo_trend',
    'home_rolling_10_goals_for', 'home_rolling_10_goals_against', 'home_rolling_10_result_points', 'home_rolling_10_form_vs_expected', 'home_rolling_10_elo_trend',
    'away_rolling_10_goals_for', 'away_rolling_10_goals_against', 'away_rolling_10_result_points', 'away_rolling_10_form_vs_expected', 'away_rolling_10_elo_trend',
    'recent_goal_total_5', 'recent_defence_total_5', 'recent_form_gap_5', 'recent_form_gap_10', 'recent_attack_gap_5', 'recent_defence_gap_5', 'rest_gap', 'activity_gap_365',
]
TOURNAMENT_LEVELS = ['world_cup', 'qualifier', 'continental', 'friendly']

def labels_of(frame):
    return frame.result.map(LABELS).to_numpy()

def base_matrix(frame):
    out = pd.DataFrame(index=frame.index)
    out['elo_difference'] = frame.elo_difference.astype(float) / 400.0
    out['neutral'] = frame.neutral.astype(float)
    return out

def rich_matrix(frame):
    out = frame[RICH_FEATURES].astype(float).copy()
    for col in ['elo_difference', 'abs_elo_difference', 'home_elo', 'away_elo', 'home_rolling_5_elo_trend', 'away_rolling_5_elo_trend', 'home_rolling_10_elo_trend', 'away_rolling_10_elo_trend']:
        out[col] = out[col] / 400.0
    out['neutral'] = frame.neutral.astype(float)
    out['rest_gap'] = out.rest_gap / 30.0
    for level in TOURNAMENT_LEVELS:
        out[f'tournament_{level}'] = (frame.tournament_type == level).astype(float)
    return out

def brier(y, probs):
    return float(np.mean(np.sum((probs - np.eye(3)[y]) ** 2, axis=1)))

def normalize(probs):
    probs = np.asarray(probs, dtype=float)
    probs = np.clip(probs, 1e-12, None)
    return probs / probs.sum(axis=1, keepdims=True)

def probability_frame(model_name, probs):
    probs = normalize(probs)
    return {f'{model_name}_home_win': probs[:, 0], f'{model_name}_draw': probs[:, 1], f'{model_name}_away_win': probs[:, 2], f'{model_name}_prediction': ORDERED[probs.argmax(axis=1)]}

def sample_weights(frame):
    age_years = (frame.date.max() - frame.date).dt.days / 365.25
    recency = np.exp(-age_years / 8.0)
    type_weight = frame.tournament_type.map({'world_cup': 1.5, 'qualifier': 1.25, 'continental': 1.15, 'friendly': 0.7, 'other': 1.0}).fillna(1.0)
    return np.asarray(recency * type_weight, dtype=float)

def poisson_outcomes(home_rate, away_rate, cap=10):
    home = np.array([exp(-home_rate) * home_rate ** k / factorial(k) for k in range(cap + 1)])
    away = np.array([exp(-away_rate) * away_rate ** k / factorial(k) for k in range(cap + 1)])
    matrix = np.outer(home, away)
    matrix /= matrix.sum()
    return np.array([np.tril(matrix, -1).sum(), np.trace(matrix), np.triu(matrix, 1).sum()])

def fit_poisson(train, test, matrix_func=base_matrix):
    X_train, X_test = matrix_func(train), matrix_func(test)
    home = PoissonRegressor(alpha=0.1, max_iter=1000).fit(X_train, train.home_score)
    away = PoissonRegressor(alpha=0.1, max_iter=1000).fit(X_train, train.away_score)
    rates = np.column_stack([home.predict(X_test), away.predict(X_test)])
    probs = np.vstack([poisson_outcomes(h, a) for h, a in rates])
    return probs

def fit_candidate(name, train, test):
    y = labels_of(train)
    if name == 'current_logistic_simple':
        model = make_pipeline(StandardScaler(), LogisticRegression(C=0.1, max_iter=2000, random_state=SEED)).fit(base_matrix(train), y)
        return normalize(model.predict_proba(base_matrix(test)))
    if name == 'poisson_simple':
        return normalize(fit_poisson(train, test, base_matrix))
    if name == 'elo_market_free':
        return normalize(test[['elo_home_probability', 'elo_draw_probability', 'elo_away_probability']].to_numpy(float))
    if name == 'rich_logistic':
        model = make_pipeline(StandardScaler(), LogisticRegression(C=0.2, max_iter=3000, random_state=SEED, class_weight='balanced')).fit(rich_matrix(train), y)
        return normalize(model.predict_proba(rich_matrix(test)))
    if name == 'weighted_rich_logistic':
        model = make_pipeline(StandardScaler(), LogisticRegression(C=0.2, max_iter=3000, random_state=SEED)).fit(rich_matrix(train), y, logisticregression__sample_weight=sample_weights(train))
        return normalize(model.predict_proba(rich_matrix(test)))
    if name == 'random_forest_rich':
        model = RandomForestClassifier(n_estimators=350, max_depth=7, min_samples_leaf=18, random_state=SEED, n_jobs=-1, class_weight='balanced_subsample').fit(rich_matrix(train), y)
        return normalize(model.predict_proba(rich_matrix(test)))
    if name == 'xgboost_rich':
        model = XGBClassifier(objective='multi:softprob', num_class=3, n_estimators=260, max_depth=2, learning_rate=0.035, subsample=0.85, colsample_bytree=0.85, reg_lambda=2.0, random_state=SEED, eval_metric='mlogloss', n_jobs=2).fit(rich_matrix(train), y, sample_weight=sample_weights(train))
        return normalize(model.predict_proba(rich_matrix(test)))
    raise ValueError(name)

def validation_split(train):
    ordered = train.sort_values(['date', 'match_id']).reset_index(drop=True)
    cut = int(len(ordered) * 0.82)
    return ordered.iloc[:cut].copy(), ordered.iloc[cut:].copy()

CANDIDATES = ['current_logistic_simple', 'poisson_simple', 'elo_market_free', 'rich_logistic', 'weighted_rich_logistic', 'random_forest_rich', 'xgboost_rich']
STACKABLE = ['elo_market_free', 'rich_logistic', 'weighted_rich_logistic', 'random_forest_rich', 'xgboost_rich']

def validation_weights(train):
    dev, val = validation_split(train)
    y_val = labels_of(val)
    losses = {}
    for name in STACKABLE:
        probs = fit_candidate(name, dev, val)
        losses[name] = log_loss(y_val, probs, labels=[0, 1, 2])
    raw = np.exp(-3.0 * (pd.Series(losses) - min(losses.values())))
    weights = raw / raw.sum()
    return weights.to_dict(), losses

def ensemble_from_weights(probs_by_model, weights):
    total = sum(weights[name] * probs_by_model[name] for name in weights)
    return normalize(total)

assert rich_matrix(featured.head()).shape[1] > base_matrix(featured.head()).shape[1]
print('Feature and model helpers ready.')


Feature and model helpers ready.


In [3]:
prediction_frames = []
metric_rows = []
weight_rows = []

for year in YEARS:
    tournament = featured[(featured.tournament == 'FIFA World Cup') & (featured.date.dt.year == year)].copy()
    assert len(tournament) == 64, f'{year} has {len(tournament)} World Cup rows, expected 64'
    cutoff = tournament.date.min()
    train = featured[(featured.date < cutoff) & (featured.date >= '1990-01-01')].copy()
    current_train = featured[(featured.date < cutoff) & (featured.date >= '2000-01-01')].copy()
    assert train.date.max() < cutoff and not set(train.match_id) & set(tournament.match_id)

    weights, validation_losses = validation_weights(train)
    for name, value in weights.items():
        weight_rows.append({'world_cup': year, 'model': name, 'validation_log_loss': validation_losses[name], 'ensemble_weight': value})

    probs_by_model = {}
    for name in CANDIDATES:
        train_for_model = current_train if name in ['current_logistic_simple', 'poisson_simple'] else train
        probs_by_model[name] = fit_candidate(name, train_for_model, tournament)

    probs_by_model['validation_weighted_ensemble'] = ensemble_from_weights(probs_by_model, weights)
    probs_by_model['equal_ml_ensemble'] = normalize(np.mean([probs_by_model[name] for name in STACKABLE], axis=0))

    actual = labels_of(tournament)
    for name, probs in probs_by_model.items():
        metric_rows.append({'world_cup': year, 'model': name, 'matches': len(tournament), 'log_loss': log_loss(actual, probs, labels=[0, 1, 2]), 'brier': brier(actual, probs), 'accuracy': float((probs.argmax(axis=1) == actual).mean())})

    out = tournament[['match_id', 'date', 'home_team', 'away_team', 'home_score', 'away_score', 'result', 'wc_group_stage', 'wc_knockout_stage']].copy()
    out.insert(0, 'world_cup', year)
    for name, probs in probs_by_model.items():
        for key, value in probability_frame(name, probs).items():
            out[key] = value
    prediction_frames.append(out)

predictions = pd.concat(prediction_frames, ignore_index=True)
metrics = pd.DataFrame(metric_rows)
weights = pd.DataFrame(weight_rows)
aggregate = metrics.groupby('model').apply(lambda g: pd.Series({
    'mean_log_loss': np.average(g.log_loss, weights=g.matches),
    'mean_brier': np.average(g.brier, weights=g.matches),
    'accuracy': np.average(g.accuracy, weights=g.matches),
    'correct': int(round(np.average(g.accuracy, weights=g.matches) * g.matches.sum())),
    'matches': int(g.matches.sum()),
}), include_groups=False).sort_values(['accuracy', 'mean_log_loss'], ascending=[False, True])

display(metrics.pivot(index='world_cup', columns='model', values='accuracy').round(4))
display(aggregate.round(4))
display(weights.pivot(index='world_cup', columns='model', values='ensemble_weight').round(3))

assert predictions.groupby('world_cup').size().to_dict() == {year: 64 for year in YEARS}
assert len(predictions) == 256
for name in list(probs_by_model):
    cols = [f'{name}_home_win', f'{name}_draw', f'{name}_away_win']
    assert np.allclose(predictions[cols].sum(axis=1), 1.0)
assert aggregate.loc['current_logistic_simple', 'accuracy'] == 0.5546875

REPORTS.mkdir(exist_ok=True)
predictions.to_csv(REPORTS / 'world_cup_accuracy_improvement_predictions.csv', index=False)
metrics.to_csv(REPORTS / 'world_cup_accuracy_improvement_metrics.csv', index=False)
weights.to_csv(REPORTS / 'world_cup_accuracy_improvement_ensemble_weights.csv', index=False)

accuracy_table = metrics.pivot(index='world_cup', columns='model', values='accuracy').round(4).to_csv()
logloss_table = metrics.pivot(index='world_cup', columns='model', values='log_loss').round(4).to_csv()
summary_table = aggregate.round(4).to_csv()
weight_table = weights.pivot(index='world_cup', columns='model', values='ensemble_weight').round(4).to_csv()
best_model = aggregate.index[0]
best = aggregate.loc[best_model]
baseline = aggregate.loc['current_logistic_simple']
delta = best.accuracy - baseline.accuracy
report = f'''# World Cup Accuracy Improvement Experiments\n\nThis notebook tested richer market-free methods on the same frozen 2010-2022 World Cup audit used by the original report: 64 matches per tournament, 256 total. Predictions for each tournament are trained only on matches before that tournament's opening date.\n\n## Data availability\n\nAvailable in-repo data supports Elo, match results, tournament labels, neutral-site flags, dates, scores, rolling team form, inferred World Cup group/knockout stage flags, and recency/tournament weighting. The repo does not contain betting odds, squad lists, player availability, injuries, player market values, or FIFA ranking history, so those methods were not fabricated here.\n\n## Aggregate ranking\n\n```csv\n{summary_table}```\n\nBest accuracy model: `{best_model}` with {int(best.correct)}/{int(best.matches)} correct ({best.accuracy:.2%}). Original simple logistic reference: {int(baseline.correct)}/{int(baseline.matches)} correct ({baseline.accuracy:.2%}). Accuracy delta: {delta:+.2%}.\n\n## Accuracy by World Cup\n\n```csv\n{accuracy_table}```\n\n## Log loss by World Cup\n\n```csv\n{logloss_table}```\n\n## Validation-derived ensemble weights\n\n```csv\n{weight_table}```\n\n## Interpretation\n\nThe experiment tries the feasible improvement ideas from local data: more historical training data from 1990 onward, leakage-safe rolling form windows, draw/strength-similarity features, inferred group-vs-knockout flags, recency and tournament-type sample weighting, random forest, XGBoost, and simple probability ensembles. Accuracy remains a noisy secondary metric for football 1X2 outcomes; log loss and Brier score still matter because they measure probability quality rather than only the top class.\n\nPer-match predictions are saved to `reports/world_cup_accuracy_improvement_predictions.csv`; per-model metrics are saved to `reports/world_cup_accuracy_improvement_metrics.csv`.\n'''
(REPORTS / 'world_cup_accuracy_improvement_experiments.md').write_text(report, encoding='utf-8')
print(f'Best: {best_model} {int(best.correct)}/{int(best.matches)} accuracy={best.accuracy:.4f}, delta={delta:+.4f}')
print('All improvement experiment checks passed.')


G:\Coding\nations-ai\.venv\Lib\site-packages\joblib\externals\loky\backend\context.py:131: UserWarning: Could not find the number of physical cores for the following reason:
found 0 physical cores < 1
Returning the number of logical cores instead. You can silence this warning by setting LOKY_MAX_CPU_COUNT to the number of cores you want to use.
  warnings.warn(
  File "G:\Coding\nations-ai\.venv\Lib\site-packages\joblib\externals\loky\backend\context.py", line 255, in _count_physical_cores
    raise ValueError(f"found {cpu_count_physical} physical cores < 1")


model,current_logistic_simple,elo_market_free,equal_ml_ensemble,poisson_simple,random_forest_rich,rich_logistic,validation_weighted_ensemble,weighted_rich_logistic,xgboost_rich
world_cup,,,,,,,,,
2010,0.5156,0.5156,0.5312,0.5156,0.5312,0.5625,0.5312,0.5156,0.5156
2014,0.5781,0.5312,0.5312,0.6406,0.4375,0.4844,0.5312,0.5781,0.5938
2018,0.5781,0.5625,0.5625,0.5625,0.4688,0.5469,0.5625,0.5781,0.5781
2022,0.5469,0.5312,0.5625,0.4844,0.5156,0.5625,0.5625,0.5312,0.5469


,mean_log_loss,mean_brier,accuracy,correct,matches
model,,,,,
xgboost_rich,0.9838,0.5816,0.5586,143.0,256.0
current_logistic_simple,0.9864,0.5818,0.5547,142.0,256.0
weighted_rich_logistic,0.9780,0.5726,0.5508,141.0,256.0
poisson_simple,0.9927,0.5897,0.5508,141.0,256.0
validation_weighted_ensemble,0.9858,0.5830,0.5469,140.0,256.0
equal_ml_ensemble,0.9866,0.5836,0.5469,140.0,256.0
rich_logistic,1.0225,0.6031,0.5391,138.0,256.0
elo_market_free,0.9909,0.5884,0.5352,137.0,256.0
random_forest_rich,1.0267,0.6149,0.4883,125.0,256.0


model,elo_market_free,random_forest_rich,rich_logistic,weighted_rich_logistic,xgboost_rich
world_cup,,,,,
2010,0.193,0.186,0.189,0.216,0.216
2014,0.190,0.188,0.194,0.216,0.212
2018,0.191,0.188,0.196,0.215,0.210
2022,0.190,0.190,0.196,0.214,0.211


Best: xgboost_rich 143/256 accuracy=0.5586, delta=+0.0039
All improvement experiment checks passed.
